[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/black_sandwich/black_sandwich.ipynb)

# Black Polarization Sandwiches

A black polarization sandwich is a wave plate between crossed polarizers. One sandwich can pass light, yet two identical sandwiches in succession pass none, whatever the plate and whatever light comes in: the sandwich is a square root of zero.

The electric field of the light is modelled as a bivector of spacetime, extrinsic, and the axes of the polarizers and plates as directions, antivectors. Each axis sweeps a plane through the observer's time, `axis & time`; a polarizer reads the field's component along its axis by pairing the field with that plane, and passes it on as the field along the axis, the dual of the same plane, `(axis & time).dual()`. Each component maps fields to fields, and one stack is their composition, `sandwich = exit(plate(entrance))`: a nonzero map with `sandwich(sandwich)` zero.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from itertools import accumulate

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import show_animation
from examples.optics.black_sandwich import render

np.set_printoptions(precision=3, suppress=True)

In [ ]:
# Spacetime, space positive and time negative.
ga = Algebra("x+y+z+t-")
mv = NumpyContext(ga).multivector
# The electric field is a bivector, extrinsic; the components' axes are directions, antivectors.
Field = ga.gatype.bivector()                                     # Bivector+
Direction = ga.gatype.antivector()                               # Antivector-
# The direction in time of an observer at rest with the apparatus, which pairs to one with `mv.t`; the
# dual carries the minus of t's square.
time = -mv.t.dual()                                              # [] Direction-

## 1. Two stacks

A polarizer passes the field's component along its axis. A half-wave plate keeps the component along its fast axis and reverses the one across it, `2 * polarizer(plate_axes, plate_axes) - Field`, up to a common phase that does not affect the intensity. The plates swing from parallel to the entrance polarizer to ninety degrees and back.

In the animation light passes from left to right through two identical stacks. Dark lines mark the polarizers' axes and orange lines the plates'; above the beam is the field between components, and the beam's strength follows the intensity. The first stack's output always lies along its exit polarizer, which is crossed with the second stack's entrance, so the second stack stays dark throughout.

In [ ]:
n_frames = 108
duration_ms = 40
# The plates' swing out; the animation plays it forward and back.
cycle = np.linspace(0, np.pi, n_frames // 2 + 1)
plate_angles = (1 - np.cos(cycle)) * np.pi / 4
input_axis = mv.x.dual()                                         # [] Direction-
output_axis = mv.y.dual()                                        # [] Direction-

In [ ]:
# The field along a direction, the dual of the plane the direction sweeps in time: the vacuum's
# identification of a direction with the field along it, pairing to one with that plane for a unit direction.
field_along = (Direction & time).dual()                          # [] Field+ <- Direction-
# The field along one direction, weighted by the field's pairing with the plane another sweeps in time;
# bound to a unit axis twice, the polarizer that transmits the field's component along the axis.
polarizer = field_along * ((Direction & time) & Field)           # [] Field+ <- (Direction-, Direction-, Field+)

plate_turns = (mv.xy * (-plate_angles / 2)).exp()                # [frames] Rotor+
plate_axes = plate_turns >> input_axis                           # [frames] Direction-
entrance = polarizer(input_axis, input_axis)                     # [] Field+ <- Field+
exit = polarizer(output_axis, output_axis)                       # [] Field+ <- Field+
plate = 2 * polarizer(plate_axes, plate_axes) - Field            # [frames] Field+ <- Field+
sandwich = exit(plate(entrance))                                 # [frames] Field+ <- Field+

# The field along the entrance axis, followed through both copies of the same stack.
incident = field_along(input_axis) * np.ones(len(cycle))         # [frames] Field+
components = (entrance, plate, exit) * 2
stages = accumulate(components, lambda field, component: component(field), initial=incident)
fields = stack(tuple(stages), axis=-1)                           # [frames, stages] Field+
show_animation(render.frames(fields, plate_axes, input_axis, output_axis), "black_sandwich", duration_ms,
               forward_then_back=True)

## 2. A square root of zero

With the plate at 45 degrees one stack passes the whole incident field, turned onto the exit axis. Two stacks pass nothing, for every input: the stack's whole image lies in its kernel. This holds whatever linear element sits between the same crossed polarizers, as Berry and Dennis show in [Black polarization sandwiches are square roots of zero](https://michaelberryphysics.wordpress.com/wp-content/uploads/2013/07/berry361.pdf).

In the Jones calculus the stack reads as the matrix product $P_y H P_x$, a $2\times 2$ matrix with a single nonzero entry off the diagonal, and its square is the zero matrix.

In [ ]:
brightest = n_frames // 4
single_stack = sandwich[brightest]                               # [] Field+ <- Field+
double_stack = single_stack(single_stack)                        # [] Field+ <- Field+

In [ ]:
print("One stack")
print(single_stack.coefficient_table())
print("Two stacks")
print(double_stack.coefficient_table())

In [ ]:
# checks
np.testing.assert_allclose((fields[:, 3] & (output_axis & time)).to_array().max(), 1, atol=1e-12)
np.testing.assert_allclose(sandwich(sandwich).kernel, 0, atol=1e-12)
np.testing.assert_allclose(fields[:, 4:].kernel, 0, atol=1e-12)